# AI Agent for FP&A — Query Handler Example

Este notebook demonstra um cliente simples para o Databricks Genie, com fallback local e geração de relatórios sintéticos para testes.

## Contexto

O objetivo: um analista pergunta em linguagem natural e recebe uma resposta baseada nas tabelas gold. Em ambiente local o notebook gera respostas sintéticas para demonstração.

In [ ]:
import requests
import os
import logging
from typing import Dict, Any, Optional
import pandas as pd

# Configuração opcional (quando executado no Databricks setar as ENV vars)

In [ ]:
DATABRICKS_HOST = os.getenv("DATABRICKS_HOST")
DATABRICKS_TOKEN = os.getenv("DATABRICKS_TOKEN")
GENIE_SPACE_ID = os.getenv("GENIE_SPACE_ID")  # criado no UI do Databricks

HEADERS = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}" if DATABRICKS_TOKEN else "",
    "Content-Type": "application/json"
}

# Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("genie_query_handler")

In [ ]:
def ask_genie(question: str, space_id: str) -> dict:
    """
    Envia uma pergunta para o espaço Genie e retorna a resposta (ou fallback sintético).
    """
    url = f"{DATABRICKS_HOST}/api/2.0/genie/spaces/{space_id}/messages"
    payload = {
        "question": question,
        "generation_config": {"max_tokens": 500, "temperature": 0.1}
    }
    if not DATABRICKS_HOST or not DATABRICKS_TOKEN:
        logger.info("Databricks credentials not found — returning synthetic response")
        synthetic = {
            "question": question,
            "answer": "Synthetic answer: total IT cost for May 2025 was BRL 1,234,567.89",
            "sql_query": "SELECT SUM(amount) FROM gold.monthly_total WHERE month = '2025-05-01'",
            "sources": ["gold.monthly_total"]
        }
        return synthetic
    response = requests.post(url, json=payload, headers=HEADERS)
    response.raise_for_status()
    result = response.json()
    return {
        "question": question,
        "answer": result.get("answer"),
        "sql_query": result.get("generated_sql"),
        "sources": result.get("used_data_sources")
    }

## Prompt Engineering

Defina o `GENIE_SYSTEM_PROMPT` no espaço Genie para restringir o escopo e formato das respostas.

In [ ]:
GENIE_SYSTEM_PROMPT = """
You are a financial data assistant for the IT Cost Analytics team.
You ONLY answer questions about IT costs, headcount, and vendor spend.
Rules: Always include the SQL used; if ambiguous ask for clarification; use BRL format.
""".strip()

In [ ]:
def log_query(question: str, answer: str, sql: str, analyst: str):
    """Logs para auditoria local (escreve CSV em reports/)."""
    log_entry = {
        "timestamp": pd.Timestamp.now(),
        "question": question,
        "answer": answer,
        "sql_generated": sql,
        "analyst": analyst,
        "validated": False
    }
    audit_path = os.path.join(os.path.dirname(__file__), '..', '..', 'reports', 'genie_query_audit.csv')
    os.makedirs(os.path.dirname(audit_path), exist_ok=True)
    df = pd.DataFrame([log_entry])
    if not os.path.exists(audit_path):
        df.to_csv(audit_path, index=False)
    else:
        df.to_csv(audit_path, mode="a", header=False, index=False)
    logger.info("Query logged for review: %s", question)

In [ ]:
def generate_sample_report() -> pd.DataFrame:
    """Gera um relatório sintético por categoria e mês e salva em reports/."""
    months = pd.date_range(start="2024-06-01", periods=12, freq='MS')
    categories = ["SOFTWARE", "HARDWARE", "CLOUD", "CONSULTING"]
    rows = []
    for m in months:
        for c in categories:
            rows.append({
                "month": m.strftime("%Y-%m-%d"),
                "category": c,
                "amount_brl": round(100000 * (1 + 0.1 * (categories.index(c))) * (1 + 0.02 * (months.get_loc(m))), 2)
            })
    df = pd.DataFrame(rows)
    report_path = os.path.join(os.path.dirname(__file__), '..', '..', 'reports', 'genie_sample_report.csv')
    os.makedirs(os.path.dirname(report_path), exist_ok=True)
    df.to_csv(report_path, index=False)
    logger.info("Synthetic Genie report written to %s", report_path)
    return df

## Demo

Executar a célula abaixo simula uma pergunta e gera os artefatos em `reports/`.

In [ ]:
q = "What was the total IT cost for May 2025?"
res = ask_genie(q, GENIE_SPACE_ID or "demo_space")
print("Answer:", res.get("answer"))
log_query(res.get("question"), res.get("answer"), res.get("sql_query"), analyst="local_user")
generate_sample_report()